# ENERGICAL — Client Loyalty & Retention Model
### DataFest 2026 | The Outliers | June 2026

**Objective:** Predict client retention probability and generate personalized commercial strategies per wilaya segment.


## 0. Imports & Configuration

In [41]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from scipy.stats import chi2_contingency, ttest_ind

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.impute import SimpleImputer

import statsmodels.api as sm

## 1 · Data Loading

In [42]:
transactions = pd.read_csv(
    r"C:\Users\PCPRODZ\Desktop\energical-datasci\Data\data_clean.csv")
products = pd.read_csv(
    r"C:\Users\PCPRODZ\Downloads\LIVRABLE_DataFest2026_ENERGICAL\LIVRABLE_etudiants\energical_catalogue_produits.csv"       
)

###  Findings
- Transactions: 17,263 rows × 13 columns (2022–2024)
- Catalogue: 731 products × 10 columns
- Join key: `produit` (transactions) ↔ `nom_produit` (catalogue)

## 3. Feature engineering

### 2.1 · Merge transactions with catalogue

Left join to enrich transactions 

In [43]:
transactions["produit"] = transactions["produit"].str.strip()
products["nom_produit"] = products["nom_produit"].str.strip()

df = transactions.merge(
    products,
    left_on="produit",
    right_on="nom_produit",
    how="left"
)

### 2.2 · Compute estimated profit

`profit_estime = montant_da × marge_estimee_pct / 100`

Per-transaction profit estimate in DA — used later to score client economic value.

In [44]:
df["profit_estime"] = (
    df["montant_da"] *
    df["marge_estimee_pct"]/100
)

KeyError: 'marge_estimee_pct'

### 2.3 · Aggregate to client level

Each row in `customer_df` = 1 unique client. Features built:
- **Behavioral:** total orders, total spent, avg order, total quantity
- **Preference:** favorite category, favorite payment, most frequent wilaya
- **Economic:** avg margin, avg price, avg profit, avg restock delay

In [ ]:
customer_df = (
    df.groupby("id_client")
      .agg(
          returning=("nouveau_ou_fidele", "first"),

          wilaya=("wilaya",
                  lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan),

          type_client=("type_client", "first"),

          favorite_category=("categorie_produit",
                  lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan),

          favorite_payment=("moyen_paiement",
                  lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan),

          total_orders=("id_transaction","count"),
          total_spent=("montant_da","sum"),
          average_order=("montant_da","mean"),
          total_quantity=("quantite","sum"),
          average_margin=("marge_estimee_pct","mean"),
          average_price=("prix_da","mean"),
          average_profit=("profit_estime","mean"),
          average_restock=("delai_reappro_jours","mean")
      )
      .reset_index()
)

### 2.4 · Filter for retention analysis + compute recency

Drop rows missing key fields, then compute `days_since_last_order` — the primary churn signal.

In [ ]:
df_retention = df.copy()

df_retention = df_retention.dropna(
    subset=[
        "produit",
        "quantite",
        "montant_da",
        "moyen_paiement"
    ]
)

In [ ]:
customer_df["average_quantity_per_order"] = (
    customer_df["total_quantity"] /
    customer_df["total_orders"]
)

df_retention["date_commande"] = pd.to_datetime(
    df_retention["date_commande"],
    dayfirst=True,
    errors="coerce"
)

reference_date = df_retention["date_commande"].max()

recency = (
    df_retention.groupby("id_client")["date_commande"]
    .max()
)

customer_df["days_since_last_order"] = (
    reference_date - customer_df["id_client"].map(recency)
).dt.days

## 3 · Target Variable & Preprocessing

### 3.1 · Define target variable and features

**Target:** `Returning` — 1 if the client is a returning customer, 0 if new.

**Features:**
- Categorical: `wilaya`, `type_client`, `favorite_category`, `favorite_payment`
- Numerical: `total_orders`, `total_spent`, `average_order`, `total_quantity`, `average_quantity_per_order`, `days_since_last_order`

In [ ]:
customer_df["Returning"] = (
    customer_df["returning"] == "Returning"
).astype(int)

features = [
    "wilaya",
    "type_client",
    "favorite_category",
    "favorite_payment",
    "total_orders",
    "total_spent",
    "average_order",
    "total_quantity",
    "average_quantity_per_order",
    "days_since_last_order"
]

X = customer_df[features]
y = customer_df["Returning"]

### 3.2 · Train / test split (80/20, stratified)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

### 3.3 · Preprocessing pipeline

- **Numerical:** median imputation
- **Categorical:** most-frequent imputation → one-hot encoding (`handle_unknown='ignore'`)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

categorical = [
    "wilaya",
    "type_client",
    "favorite_category",
    "favorite_payment"
]

numerical = [
    "total_orders",
    "total_spent",
    "average_order",
    "total_quantity",
    "average_quantity_per_order",
    "days_since_last_order"
]

preprocessor = ColumnTransformer([
    (
        "num",
        SimpleImputer(strategy="median"),
        numerical
    ),
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]),
        categorical
    )
])

## 4 · Logistic Regression (Baseline)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

model = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

### 4.1 · Evaluation

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

pred = model.predict(X_test)
prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, pred))
print(confusion_matrix(y_test, pred))
print("ROC AUC:", roc_auc_score(y_test, prob))

In [ ]:
customer_df["Retention Score"] = model.predict_proba(X)[:, 1]

In [ ]:
def segment(score):
    if score >= 0.80:
        return "High"

    elif score >= 0.50:
        return "Medium"

    return "Low"

customer_df["Retention Level"] = customer_df["Retention Score"].apply(segment)

In [ ]:
def strategy(row):

    if row["Retention Score"] >= 0.80:
        return "Maintain loyalty through VIP rewards and exclusive offers."

    if row["days_since_last_order"] > 180:
        return "Launch a win-back campaign with personalized promotions."

    if row["total_orders"] <= 2:
        return "Offer a second-purchase discount to encourage repeat buying."

    if row["average_order"] < customer_df["average_order"].median():
        return "Promote bundles or complementary products to increase basket size."

    return "Send personalized recommendations based on the customer's preferred category."

customer_df["Recommended Strategy"] = customer_df.apply(strategy, axis=1)

## 5 · Random Forest (Final Model)

300 trees. Handles non-linearity, high-cardinality categoricals (48 wilayas), and class imbalance better than Logistic Regression.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = Pipeline([
    ("prep", preprocessor),
    ("clf", RandomForestClassifier(
        n_estimators=300,
        random_state=42
    ))
])

rf_model.fit(X_train, y_train)

### 5.1 · Feature importances



In [ ]:
feature_names = rf_model.named_steps["prep"].get_feature_names_out()

In [ ]:
importances = rf_model.named_steps["clf"].feature_importances_

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
})

importance_df = importance_df.sort_values(
    "Importance",
    ascending=False
)

print(importance_df.head(20))

###  Findings
- `total_spent`, `total_orders`, `average_order` = top 3 predictors — high-value clients return more
- `days_since_last_order` = 5th — recency matters but volume dominates
- Wilaya (Alger, Tlemcen, Oran) and category (Électricité, Sanitaire) have moderate weight
- **Business implication:** target clients with high historical spend but growing inactivity

## 7 · Export


In [ ]:
cols_to_export = [
    "id_client", "wilaya", "type_client",
    "favorite_category", "favorite_payment",
    "total_orders", "total_spent", "average_order",
    "total_quantity", "average_quantity_per_order",
    "days_since_last_order",
    "Retention Score", "Retention Level", "Recommended Strategy"
]

customer_df[cols_to_export].to_csv("customer_loyalty.csv", index=False)
print(f"✅ Exported customer_loyalty.csv — {len(customer_df)} clients")
print(customer_df["Retention Level"].value_counts())